In [1163]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [1164]:
script_dir = os.getcwd()
parent_dir = os.path.dirname(script_dir)
env_path = os.path.join(parent_dir, ".env")
load_dotenv(env_path)

os.makedirs('./files', exist_ok=True)

db_info = { 
    'host': os.getenv("DB_HOST", "localhost"),
    'user': os.getenv("DB_USER", "root"),
    'password': os.getenv("DB_PASSWORD"),
    'name': os.getenv("DB_NAME")
}

Retrieving all nikke data from the database

In [1165]:
#database setup and connection
engine = create_engine(f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}/{db_info['name']}")

#query to grab all nikkes post launch and their metadata I want to train on
post_launch_nikkes_query = """
    select 
        characters.name,
        characters.class,
        characters.weapon,
        characters.manufacturer,
        characters.element,
        characters.burst,
        characters.overspec,
        release_history.start_date
    from release_history 
    join characters on release_history.name = characters.name
    where release_history.start_date > '2022-11-04'
    order by release_history.start_date asc
"""

df = pd.read_sql(post_launch_nikkes_query, con=engine)
print(len(df))

112


Encoding the nikke data, and setting event flags, and 3d time spiral

In [1166]:
traits = ['class', 'weapon', 'manufacturer', 'element', 'burst']
#one hot encoding
df_encoded = pd.get_dummies(df, columns=traits, dtype=int) #get dummies takes every unique value in a column and expand it into its own column, and gives u true and false for each row
df_encoded['start_date'] = pd.to_datetime(df_encoded['start_date'])

days = df_encoded['start_date'].dt.dayofyear
months = df_encoded['start_date'].dt.month

raw_days = (df_encoded['start_date'] - pd.to_datetime('2022-11-04')).dt.days #gives you a df of days since launch for each nikke
#will use a normalization of day - min(day) / max(day) - min(day) day = days since launch to normalize the days since launch. so 700 days becomes like .78

df_encoded['cos_day'] = np.cos((days/365.25) * 2 * np.pi) #maps every single day of the year into a 2d circle
df_encoded['sin_day'] = np.sin((days/365.25) * 2 * np.pi)
df_encoded['days_since_launch'] = (raw_days - raw_days.min()) / (raw_days.max() - raw_days.min()) #calculates the number days since launch. Helps turn the 2d spiral of dates into a 3d spiral 
df_encoded['is_anniversary'] = months.isin([4,5,10,11]).astype(int) #marks true if a unit was released during anniversary time, ect.
df_encoded['is_new_years'] = months.isin([12,1]).astype(int)
df_encoded['is_summer'] = months.isin([6,7]).astype(int)

# print(df_encoded[df_encoded['name'] == 'Red Hood'])
df_encoded = df_encoded.drop(columns=['start_date'])

base_traits = ['name', 'cos_day', 'sin_day', 'days_since_launch', 'is_anniversary', 'is_new_years', 'is_summer', 'overspec']

for trait in traits: 
    trait_cols = [col for col in df_encoded.columns if col.startswith(f"{trait}_")]
    trait_cols = base_traits + trait_cols
    
    df_split = df_encoded[trait_cols]

    filename = f"./files/lstm_train_{trait}.csv"
    df_split.to_csv(filename, index=False)
    print(f"Saved {filename} with columns: {trait_cols}")

filename = f"./files/lstm_train_MTL.csv"
df_encoded.to_csv(filename, index=False)
print(f"Saved {filename} with columns: {df_encoded.columns}")

Saved ./files/lstm_train_class.csv with columns: ['name', 'cos_day', 'sin_day', 'days_since_launch', 'is_anniversary', 'is_new_years', 'is_summer', 'overspec', 'class_Attacker', 'class_Defender', 'class_Supporter']
Saved ./files/lstm_train_weapon.csv with columns: ['name', 'cos_day', 'sin_day', 'days_since_launch', 'is_anniversary', 'is_new_years', 'is_summer', 'overspec', 'weapon_Assault Rifle', 'weapon_Minigun', 'weapon_Rocket Launcher', 'weapon_SMG', 'weapon_Shotgun', 'weapon_Sniper Rifle']
Saved ./files/lstm_train_manufacturer.csv with columns: ['name', 'cos_day', 'sin_day', 'days_since_launch', 'is_anniversary', 'is_new_years', 'is_summer', 'overspec', 'manufacturer_Abnormal', 'manufacturer_Elysion', 'manufacturer_Missilis', 'manufacturer_Pilgrim', 'manufacturer_Tetra']
Saved ./files/lstm_train_element.csv with columns: ['name', 'cos_day', 'sin_day', 'days_since_launch', 'is_anniversary', 'is_new_years', 'is_summer', 'overspec', 'element_Electric', 'element_Fire', 'element_Iron', 

Model setup

In [1167]:
class NikkePredictorMTL(nn.Module):
    def __init__(self, input_features, hidden_size=32, num_layers=1, dropout_prob=0.2):
        """
        Creates the initial weights and biases, and shared LSTM engine
        """
        super(NikkePredictorMTL, self).__init__() #register my model as a pytorch model so i can run backpropagation on it
        self.shared_lstm = nn.LSTM( #creates the forget, candidate, input, and ouput gates
            input_size = input_features,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True
        )
        self.dropout = nn.Dropout(p=dropout_prob)

        #takes the shared lstm hidden state and does affine transformation to turn it into logits or a vector for the attribute
        self.head_class = nn.Linear(hidden_size, 3) #attacker, defender, supporter
        self.head_weapon = nn.Linear(hidden_size, 6) #minigun, SR, RL, AR, SMG, SG
        self.head_element = nn.Linear(hidden_size, 5) #fire, electric, wind, iron, water
        self.head_burst = nn.Linear(hidden_size, 4) #1,2,3,all
        self.head_manufacturer = nn.Linear(hidden_size, 5) #abnormal, elysion, missilis, tetra, pilgrim
    
    def forward(self, x):
        """
        x = tensor(batch_size, sequence_length, input_features)
        Takes the output of the shared_lstm, feeds it to the attribute heads, and returns the logits for each attribute head
        """
        lstm_out, (hidden_state, cell_state) = self.shared_lstm(x)

        final_thought = self.dropout(hidden_state[-1]) #hidden state is num_layers, batch_size, hidden_size

        out_class = self.head_class(final_thought)
        out_weapon = self.head_weapon(final_thought)
        out_element = self.head_element(final_thought)
        out_burst = self.head_burst(final_thought)
        out_manufacturer = self.head_manufacturer(final_thought)

        return {
            'class': out_class,
            'weapon': out_weapon,
            'element': out_element,
            'burst': out_burst,
            'manufacturer': out_manufacturer
        }

    def predict(self, x): 
        """
        runs forward pass of input, and applies softmax to output
        """

        self.eval() #set model to evaluation mode

        with torch.no_grad(): #disable gradient calculation for faster output
            logits_dict = self.forward(x)

            return {
                'class': torch.softmax(logits_dict['class'], dim=1),
                'weapon': torch.softmax(logits_dict['weapon'], dim=1),
                'element': torch.softmax(logits_dict['element'], dim=1),
                'burst': torch.softmax(logits_dict['burst'], dim=1),
                'manufacturer': torch.softmax(logits_dict['manufacturer'], dim=1)
            }

Converting data into a sequence of tensors

In [1168]:
class NikkeSequenceDataset(Dataset):
    def __init__(self, csv_path, seq_length = 5):
         """
         Grabs all feature columns, and cleans dataset, and sets the attribute labels
         """
         self.seq_length = seq_length
         self.df = pd.read_csv(csv_path)

         self.feature_cols = [col for col in self.df.columns if col != 'name']
         self.features_df = self.df[self.feature_cols].values.astype(np.float32)

         self.class_labels = self.df[['class_Attacker', 'class_Defender', 'class_Supporter']].values.argmax(axis=1) #[0,1,2], maps attacker to 0, defender to 1, supporter to 2
         self.weapon_labels = self.df[['weapon_Assault Rifle', 'weapon_Minigun', 'weapon_Rocket Launcher', 
                                     'weapon_SMG', 'weapon_Shotgun', 'weapon_Sniper Rifle']].values.argmax(axis=1)
         self.mfg_labels = self.df[['manufacturer_Abnormal', 'manufacturer_Elysion', 'manufacturer_Missilis', 
                                  'manufacturer_Pilgrim', 'manufacturer_Tetra']].values.argmax(axis=1)
         self.element_labels = self.df[['element_Electric', 'element_Fire', 'element_Iron', 
                                      'element_Water', 'element_Wind']].values.argmax(axis=1)
         self.burst_labels = self.df[['burst_1', 'burst_2', 'burst_3', 'burst_All']].values.argmax(axis=1)

    def __len__(self):
         """
         Total number of valid sliding window sequences. i.e. 1,2,3,4 test 5, 2,3,4,5,test 6...
         """
         return len(self.features_df) - self.seq_length

    def __getitem__(self, idx):
         """
         Prepares an input sequence of x and a target prediction dictionary of y
         """
         #training set of input x stays with encoding as they are all categorical values. supporter is not greater than defender
         x = self.features_df[idx : idx + self.seq_length] #if idx = 0, its features_df[0:5], so x input is 0,1,2,3,4. Tensor shape of (5,30), for 5 characters and their 30 attributes each
         target_idx = idx + self.seq_length #target which is the test input where you guess if the model predict right, its 0 + 5, so index 5

         y = { #target dictionary must be in label vectors for loss function to calculate
            'class': torch.tensor(self.class_labels[target_idx], dtype=torch.long), #tensor(1) or smth
            'weapon': torch.tensor(self.weapon_labels[target_idx], dtype=torch.long), 
            'element': torch.tensor(self.element_labels[target_idx], dtype=torch.long),
            'burst': torch.tensor(self.burst_labels[target_idx], dtype=torch.long) ,
            'manufacturer': torch.tensor(self.mfg_labels[target_idx], dtype=torch.long)
         } #when we do cross entropy, for our logits of [int, int, int], we need to know which index is the right answer for the attribute. 
         #ex: class's logits after affine is [2,4,5], the nikke is a supporter, so the label in the dictionary should be 2 for the index of 2.
         return torch.tensor(x, dtype=torch.float32), y

In [1169]:
# def get_class_weights(df, columns, gamma = 0.001): 
#     """
#     Computes normalized inverse class frequency weights for Crossentropyloss
#     1/number of a type of attribute. less frequent, higher weight
#     """

#     counts = df[columns].sum().values.astype(np.float32) #counts up each of the column types, so [#attacker, #defender, #supporter]
#     counts = np.maximum(counts, 1.0) #makes sure none of the counts are 0 

#     inv_freq = (1.0 / counts) ** gamma #inverts all the counts and soften the weights so gradients don't explode
#     normalized_weights = inv_freq / inv_freq.sum() * len(counts) #multiplying by len(counts) # number of types of attributes,  makes it so that the average weight is around 1.0, that way it won't affect learning rate by being lower than 1

#     return torch.tensor(normalized_weights, dtype=torch.float32)


dataset = NikkeSequenceDataset('./files/lstm_train_MTL.csv', seq_length=6) #initialize dataset
total_sequences = len(dataset)

train_size = int(total_sequences * 0.90)
test_size = total_sequences - train_size

train_indices = list(range(0,train_size))
test_indices = list(range(train_size, total_sequences))

train_dataset = Subset(dataset, train_indices)
test_dataset = Subset(dataset, test_indices)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"Total Sequences: {total_sequences}")
print(f"Training Windows: {len(train_dataset)} | Testing Windows: {len(test_dataset)}")

Total Sequences: 106
Training Windows: 95 | Testing Windows: 11


Create weights

In [1170]:
"""
Penalization for rare types do not work for my dataset. it's too small, not enough actual data for the model to realize its rare. Ends up commanding all of the gradients because of the penalty
"""

# df_raw = pd.read_csv('./files/lstm_train_MTL.csv')

# class_cols = ['class_Attacker', 'class_Defender', 'class_Supporter']
# weapon_cols = ['weapon_Assault Rifle', 'weapon_Minigun', 'weapon_Rocket Launcher', 'weapon_SMG', 'weapon_Shotgun', 'weapon_Sniper Rifle']
# element_cols = ['element_Electric', 'element_Fire', 'element_Iron', 'element_Water', 'element_Wind']
# burst_cols = ['burst_1', 'burst_2', 'burst_3', 'burst_All']
# mfg_cols = ['manufacturer_Abnormal', 'manufacturer_Elysion', 'manufacturer_Missilis', 'manufacturer_Pilgrim', 'manufacturer_Tetra']

# weight_class = get_class_weights(df_raw, class_cols)
# weight_weapon = get_class_weights(df_raw, weapon_cols)
# weight_element = get_class_weights(df_raw, element_cols)
# weight_burst = get_class_weights(df_raw, burst_cols)
# weight_mfg = get_class_weights(df_raw, mfg_cols)

# print(weight_class)

"\nPenalization for rare types do not work for my dataset. it's too small, not enough actual data for the model to realize its rare. Ends up commanding all of the gradients because of the penalty\n"

Create model

In [1171]:
torch.manual_seed(44)
input_features = len(dataset.feature_cols)
#smaller hidden_size is good because alot of columns are just 0s encoded due to the attribute encoding
hidden_size = 24 #best for generalization since my dataset is very small 93 training, so having a size of 24 allows the lstm to train 56 params per sample, reduces overfitting param = 4 * (H * F + H^2 + H), H is hidden size, F is input features.
num_layers = 1
epochs = 160
best_test_loss = float('inf')

model = NikkePredictorMTL(input_features=input_features, hidden_size=hidden_size, num_layers=num_layers, dropout_prob=0.31) #Randomly set 31% of hidden state output to 0, makes it so that there are less memorization of specific pairs. the hidden size won't have co-adaptations

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-9) #weight decay keeps the weight from getting too large
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

Training Loop

In [1172]:
print("Starting Training Loop...\n")

for epoch in range(1, epochs + 1): 
    model.train()
    running_train_loss = 0.0
    for batch_x, batch_y in train_loader: 
        optimizer.zero_grad() #reset old gradients
        predictions = model(batch_x) #where does this come from? 

        loss_class = criterion(predictions['class'], batch_y['class'])
        loss_weapon = criterion(predictions['weapon'], batch_y['weapon'])
        loss_element = criterion(predictions['element'], batch_y['element'])
        loss_burst = criterion(predictions['burst'], batch_y['burst'])
        loss_mfg = criterion(predictions['manufacturer'], batch_y['manufacturer'])

        total_loss = (
            loss_class * 1.0 + 
            loss_weapon * 1.0 + 
            loss_element * 1.0 + 
            loss_burst * 1.0 + 
            loss_mfg * 1.0) #multiply the loss for underperforming attribute heads

        total_loss.backward()
        optimizer.step()
        running_train_loss += total_loss.item()

    avg_loss = running_train_loss / len(train_loader)
    scheduler.step()

    if epoch % 5 == 0 or epoch == 1: 
        model.eval()

        correct_class = 0
        correct_weapon = 0
        correct_element = 0
        correct_burst = 0
        correct_manufacturer = 0
        running_test_loss = 0.0
        total_test_samples = 0

        with torch.no_grad(): 
            for test_x, test_y in test_loader: 
                predictions = model(test_x)

                l_c = criterion(predictions['class'], test_y['class'])
                l_w = criterion(predictions['weapon'], test_y['weapon'])
                l_e = criterion(predictions['element'], test_y['element'])
                l_b = criterion(predictions['burst'], test_y['burst'])
                l_m = criterion(predictions['manufacturer'], test_y['manufacturer'])
                running_test_loss += (l_c + l_w + l_e + l_b + l_m).item()

                test_class = torch.argmax(predictions['class'], dim=-1)
                test_weapon = torch.argmax(predictions['weapon'], dim=-1)
                test_element = torch.argmax(predictions['element'], dim=-1)
                test_burst = torch.argmax(predictions['burst'], dim=-1)
                test_mfg = torch.argmax(predictions['manufacturer'], dim=-1)

                correct_class += (test_class == test_y['class']).sum().item()
                correct_weapon += (test_weapon == test_y['weapon']).sum().item()
                correct_element += (test_element == test_y['element']).sum().item()
                correct_burst += (test_burst == test_y['burst']).sum().item()
                correct_manufacturer += (test_mfg == test_y['manufacturer']).sum().item()
                total_test_samples += test_y['class'].size(0) # of nikkes to guess in the sample

        class_acc = (correct_class / total_test_samples) * 100
        weapon_acc = (correct_weapon / total_test_samples) * 100
        element_acc = (correct_element / total_test_samples) * 100
        burst_acc = (correct_burst / total_test_samples) * 100
        manufacturer_acc = (correct_manufacturer / total_test_samples) * 100
        avg_test_loss = running_test_loss / len(test_loader)

        print(f"Epoch [{epoch:02d}/{epochs:02d}] | Train Loss: {avg_loss:.4f} | Test Loss: {avg_test_loss:.4f} | "
            f"Test Acc -> Class: {class_acc:.1f}% | Wpn: {weapon_acc:.1f}% | "
            f"Elem: {element_acc:.1f}% | Bst: {burst_acc:.1f}% | Mfg: {manufacturer_acc:.1f}%")

        if avg_test_loss < best_test_loss: 
            best_test_loss = avg_test_loss
            torch.save(model.state_dict(), './Models/best_nikke_mtl_model.pth')
            print(f"New best test loss ({best_test_loss:.4f})! Saved model checkpoint. Epoch: {epoch}")
        print("-----------------------------------------------------------------")

        

print("\n Training and Validation Complete!")

Starting Training Loop...

Epoch [01/160] | Train Loss: 7.5703 | Test Loss: 7.4084 | Test Acc -> Class: 18.2% | Wpn: 9.1% | Elem: 27.3% | Bst: 45.5% | Mfg: 36.4%
New best test loss (7.4084)! Saved model checkpoint. Epoch: 1
-----------------------------------------------------------------
Epoch [05/160] | Train Loss: 7.3312 | Test Loss: 7.1643 | Test Acc -> Class: 54.5% | Wpn: 36.4% | Elem: 27.3% | Bst: 45.5% | Mfg: 36.4%
New best test loss (7.1643)! Saved model checkpoint. Epoch: 5
-----------------------------------------------------------------
Epoch [10/160] | Train Loss: 7.0378 | Test Loss: 6.9930 | Test Acc -> Class: 63.6% | Wpn: 36.4% | Elem: 18.2% | Bst: 45.5% | Mfg: 36.4%
New best test loss (6.9930)! Saved model checkpoint. Epoch: 10
-----------------------------------------------------------------
Epoch [15/160] | Train Loss: 6.9369 | Test Loss: 6.9623 | Test Acc -> Class: 72.7% | Wpn: 27.3% | Elem: 18.2% | Bst: 45.5% | Mfg: 45.5%
New best test loss (6.9623)! Saved model chec

seed = 44
Sequence Length : 6 banners

Batch Size      : 8 (12 optimizer steps / epoch)

Hidden Units    : 24 (1-layer LSTM)

Dropout         : 0.31

Weight Decay    : 1e-9 (Minimal L2)

Task Multipliers: All 1.0 (Unweighted)

scheduleer: drop learning rate to 1e-6

Best Test Loss  : 6.5474 (Epoch 45 Checkpoint) on a 160 epoch total. Gentle decrease in learning rate